# Exercise 1 

In this exercise, we will work with the `spam.csv` data file. This file contains text and a label attached to each text. The goal is to clean the text data and then build classifcation models. 

### Exercise 1(a) (2 points)

Load the `pandas` and `numpy` libraries.

In [4]:
import numpy as np 
import pandas as pd 

### Exercise 1(b) (2 points)

Read the `csv` file and create a data-frame called `spam_df`.

In [5]:
spam_df = pd.read_csv('DATA/spam.csv')

### Exercise 1(c) (2 points)

Report the frequency table of `label`.

In [6]:
spam_df['label'].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

### Exercise 1(d) (2 points)

Using the `string` library, remove all the punctuation of the `text` column.

In [7]:
import string, re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

nltk.download(['stopwords', 'punkt_tab', 'wordnet',
               'averaged_perceptron_tagger_eng'])

spam_df['text_clean'] = spam_df['text'].str.translate(
    str.maketrans('', '', string.punctuation)
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\bdupey\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Exercise 1(e) (10 points)

Using the `nltk` library, remove stopwords and pronouns from `text_clean` and store in another column called `'text_clean_1'`. 

In [8]:
stop = set(stopwords.words('english'))
def drop_stops_pronouns(t):
    toks = word_tokenize(str(t).lower())
    tagged = pos_tag(toks)
    return ' '.join(w for w, tag in tagged
                    if w not in stop and tag not in ('PRP','PRP$','WP','WP$'))
spam_df['text_clean_1'] = spam_df['text_clean'].apply(drop_stops_pronouns)

### Exercise 1(f) (10 points)

Perform Lemmatization of `text_clean_1` and store the results in another column called `text_clean_2` in the `spam_df` data frame.

In [9]:
lem = WordNetLemmatizer()
spam_df['text_clean_2'] = spam_df['text_clean_1'].apply(
    lambda t: ' '.join(lem.lemmatize(w) for w in t.split())
)

### Exercise 1(g) (5 points)

Based on the results from part 1(f), we see digits in some of the text in `text_clean_2`. Remove all the digits from `text_clean_2` and store the results in another column `text_clean_3` in the `spam_df` data frame.

In [10]:
spam_df['text_clean_3'] = spam_df['text_clean_2'].str.replace(r'\d+', '', regex=True)

### Exercise 1(h) (5 points)

Using the `TfidfVectorizer` function from `sklearn.feature_extraction.text`, convert the `text_clean_3` column into a matrix of TF-IDF features. Use `max_features=30` to create the input features, and map `label` to 0-1.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(max_features=30)
X = vec.fit_transform(spam_df['text_clean_3']).toarray()
y = spam_df['label'].map({'ham': 0, 'spam': 1}).values

### Exercise 1(i) (7 points)

Using 5-fold cross-validation strategy (use `StratifiedKFold`), train and evaluate the out-of-sample performance of the `LogisticRegression` model. Use a probability threshold of 15% (i.e., classify as spam if the predicted probability is greater than or equal to 0.15) and use `recall_score` as measure of performance.

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_scores = []
for train_idx, test_idx in skf.split(X, y):
    model = LogisticRegression(max_iter=1000)
    model.fit(X[train_idx], y[train_idx])
    proba = model.predict_proba(X[test_idx])[:, 1]
    preds = (proba >= 0.15).astype(int)
    lr_scores.append(recall_score(y[test_idx], preds))
print(np.mean(lr_scores))

0.8299507829977628


### Exercise 1(j) (7 points)

Using the same cross-validation strategy from part 1(i), train and evaluate the out-of-fold performance of the `MultinomialNB` model. Use a probability threshold of 15% (i.e., classify as spam if the predicted probability is greater than or equal to 0.15) and use `recall_score` as measure of performance.

In [13]:
from sklearn.naive_bayes import MultinomialNB

nb_scores = []
for train_idx, test_idx in skf.split(X, y):
    model = MultinomialNB()
    model.fit(X[train_idx], y[train_idx])
    proba = model.predict_proba(X[test_idx])[:, 1]
    preds = (proba >= 0.15).astype(int)
    nb_scores.append(recall_score(y[test_idx], preds))
print(np.mean(nb_scores))

0.8031767337807606


### Exercise 1(k) (8 points)

Using the same cross-validation strategy from part 1(i), train and evaluate the out-of-fold performance of a deep learning model (e.g., using `tensorflow`/`keras` or `pytorch`). Use a probability threshold of 15% (i.e., classify as spam if the predicted probability is greater than or equal to 0.15) and use `recall_score` as measure of performance.

In [15]:
import tensorflow as tf

dl_scores = []
for train_idx, test_idx in skf.split(X, y):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.fit(X[train_idx], y[train_idx], epochs=20, batch_size=32, verbose=0)
    proba = model.predict(X[test_idx], verbose=0).ravel()
    preds = (proba >= 0.15).astype(int)
    dl_scores.append(recall_score(y[test_idx], preds))
print(np.mean(dl_scores))

0.8165279642058165


### Exercise 1(l) (3 points)

Based on the results from parts 1(i), 1(j) and 1(k), what model would you use to predict spam label?

I would use the logistic regression model. 